# ManoVarta Aya Continue (Private Source, No Token)

Use this when the repository is private.

What this notebook expects in Drive:
- project source archive (or copied source folder)
- Aya base model snapshot folder
- existing `aya_bundle` adapter
- DAIC transcript folder


In [ ]:
import os
import sys
import json
import shutil
import pathlib
import subprocess

from google.colab import drive
drive.mount('/content/drive')

# ---- USER CONFIG ----
# Choose one source mode: 'drive_archive' or 'drive_folder'
SOURCE_MODE = 'drive_archive'

DRIVE_ROOT = '/content/drive/MyDrive/manovarta'
DRIVE_SOURCE_ARCHIVE = f'{DRIVE_ROOT}/share/manovarta_source.tgz'
DRIVE_SOURCE_FOLDER = f'{DRIVE_ROOT}/share/multilingualChatbot'

DAIC_ROOT = f'{DRIVE_ROOT}/daic'
INIT_ADAPTER = f'{DRIVE_ROOT}/aya_bundle'
BASE_MODEL_PATH = f'{DRIVE_ROOT}/models/aya-expanse-8b'

REPORTS_DIR = f'{DRIVE_ROOT}/reports/colab_daic_continue'
EXTRACTOR_OUT = f'{DRIVE_ROOT}/outputs/extractor-aya-8b-compact-daic-continue'

# Stable defaults
EXTRACTOR_EPOCHS = 1
EXTRACTOR_BATCH_SIZE = 1
EXTRACTOR_GRAD_ACCUM = 8
EXTRACTOR_MAX_LENGTH = 1536
EXTRACTOR_SAVE_STEPS = 10
USE_4BIT = False

REPO_DIR = '/content/multilingualChatbot'
UNPACK_DIR = '/content/_manovarta_unpack'

print('Config loaded.')


In [ ]:
# Environment quick checks
subprocess.run(['nvidia-smi'], check=False)
subprocess.run(['bash', '-lc', 'df -h /content'], check=False)

required_common = [DAIC_ROOT, INIT_ADAPTER, BASE_MODEL_PATH]
missing_common = [p for p in required_common if not os.path.exists(p)]
if missing_common:
    raise FileNotFoundError('Missing required path(s): ' + ', '.join(missing_common))

if SOURCE_MODE == 'drive_archive':
    if not os.path.exists(DRIVE_SOURCE_ARCHIVE):
        raise FileNotFoundError(f'Source archive not found: {DRIVE_SOURCE_ARCHIVE}')
elif SOURCE_MODE == 'drive_folder':
    if not os.path.exists(DRIVE_SOURCE_FOLDER):
        raise FileNotFoundError(f'Source folder not found: {DRIVE_SOURCE_FOLDER}')
else:
    raise ValueError("SOURCE_MODE must be 'drive_archive' or 'drive_folder'")

print('Input paths are present.')


In [ ]:
# Materialize source repo locally (no git clone needed)
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
if os.path.exists(UNPACK_DIR):
    shutil.rmtree(UNPACK_DIR)

if SOURCE_MODE == 'drive_archive':
    pathlib.Path(UNPACK_DIR).mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(DRIVE_SOURCE_ARCHIVE, UNPACK_DIR)
    children = [p for p in pathlib.Path(UNPACK_DIR).iterdir()]
    if len(children) == 1 and children[0].is_dir():
        src_root = str(children[0])
    else:
        src_root = UNPACK_DIR
else:
    src_root = DRIVE_SOURCE_FOLDER

shutil.copytree(src_root, REPO_DIR)
os.chdir(REPO_DIR)
print('Local source ready at:', REPO_DIR)
subprocess.run(['bash', '-lc', 'ls -la | head -n 40'], check=False)


In [ ]:
# Install dependencies
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[training]'], check=True)
print('Dependencies installed.')


In [ ]:
# Aya continuation preflight (must pass)
os.chdir(REPO_DIR)

preflight_cmd = [
    sys.executable,
    'tools/run_colab_daic_continue.py',
    '--device', 'cuda',
    '--daic-root', DAIC_ROOT,
    '--init-adapter', INIT_ADAPTER,
    '--base-model-path', BASE_MODEL_PATH,
    '--reports-dir', REPORTS_DIR,
    '--extractor-output', EXTRACTOR_OUT,
    '--extractor-model', 'CohereLabs/aya-expanse-8b',
    '--preflight-only',
]

print('Running preflight...')
print(' '.join(preflight_cmd))
subprocess.run(preflight_cmd, check=True)
print('Preflight passed.')


In [ ]:
# Full Aya continuation training
os.chdir(REPO_DIR)

train_cmd = [
    sys.executable,
    'tools/run_colab_daic_continue.py',
    '--device', 'cuda',
    '--daic-root', DAIC_ROOT,
    '--init-adapter', INIT_ADAPTER,
    '--base-model-path', BASE_MODEL_PATH,
    '--reports-dir', REPORTS_DIR,
    '--extractor-output', EXTRACTOR_OUT,
    '--extractor-model', 'CohereLabs/aya-expanse-8b',
    '--extractor-epochs', str(EXTRACTOR_EPOCHS),
    '--extractor-batch-size', str(EXTRACTOR_BATCH_SIZE),
    '--extractor-grad-accum', str(EXTRACTOR_GRAD_ACCUM),
    '--extractor-max-length', str(EXTRACTOR_MAX_LENGTH),
    '--extractor-save-steps', str(EXTRACTOR_SAVE_STEPS),
    '--smoke-limit', '8',
]

if not USE_4BIT:
    train_cmd.append('--disable-extractor-4bit')

print('Running training...')
print(' '.join(train_cmd))
subprocess.run(train_cmd, check=True)
print('Training completed.')


In [ ]:
# Verify results and create one shareable bundle
summary_path = pathlib.Path(REPORTS_DIR) / 'daic_continue_summary.json'
if not summary_path.exists():
    raise FileNotFoundError(f'Summary missing: {summary_path}')

with open(summary_path, 'r', encoding='utf-8') as f:
    summary = json.load(f)
print('Summary file:', summary_path)
print(json.dumps(summary, indent=2, ensure_ascii=False)[:4000])

bundle_root = pathlib.Path(DRIVE_ROOT) / 'share' / 'aya_continue_result_bundle'
if bundle_root.exists():
    shutil.rmtree(bundle_root)
bundle_root.mkdir(parents=True, exist_ok=True)

shutil.copytree(EXTRACTOR_OUT, bundle_root / 'extractor_output')
shutil.copytree(REPORTS_DIR, bundle_root / 'reports')

zip_path = shutil.make_archive(str(bundle_root), 'zip', root_dir=str(bundle_root))
print('Result bundle:', zip_path)
